# Сравнение и выбор связки моделей

В данном ноутбуке проведен выбор оптимальной связки для модели USER2. Выбор размера лучшего размера чанков не производился, тк уже был определен в прошлом эксперименте и существенных изменений не дал. Выбран размер 40/20.

 Сравнивались меры сходства

- cosine
- dot
- Euclidean

Итоговая таблица сравнения по мерам сходства

| Конфигурация | MRR | Precision | Recall | MAP |
|-------------|-----|-----------|--------|-----|
| **USER2_cosine** | **0.7037** | **0.3200** | **0.5217** | **0.3936** |
| USER2_euclidean | 0.6230 | 0.2880 | 0.4700 | 0.3461 |
| USER2_dot | 0.4906 | 0.2147 | 0.3500 | 0.2474 |

---
**Ключевые выводы**

Лучшая мера сходства — Cosine показала лучший MRR (0.7037) среди всех трёх метрик.

Также лидирует по Precision (0.3200), Recall (0.5217) и MAP (0.3936).

USER2 хорошо справляется с запросами на авторизацию и восстановление пароля (в них MRR практически всегда 1.0)

---
**Выявленные проблемы**

Проблема с плохой обработкой запросов с опечатками остается.

Также нормализация текста в этой модели недостаточно эффективна, что показывают метрики.



# Вспомогательные функции

In [2]:
import numpy as np
import pandas as pd
import numpy as np
import json
import pandas as pd
import faiss
from typing import List, Optional, Tuple
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from IPython.display import display, Markdown
from IPython.display import display, Markdown


c:\Users\ivasi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def chunk_text(text: str, chunk_size: int = 20, overlap: int = 5) -> List[str]:
    words = text.replace("\n", " ").split()

    if chunk_size <= 0:
        raise ValueError("chunk_size должен быть положительным.")
    if overlap >= chunk_size:
        raise ValueError("overlap должен быть меньше chunk_size.")

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(words), step):
        chunk_words = words[start : start + chunk_size]
        if not chunk_words:
            continue

        chunks.append(" ".join(chunk_words))

        if start + chunk_size >= len(words):
            break

    return chunks

def buid_chunks(
    df_docs: pd.DataFrame,
    chunk_size: Optional[int] = None,
    overlap: int = 5
) -> pd.DataFrame:

    rows = []
    
    for _, row in df_docs.iterrows():
        doc_id = row['doc_id']
        topic = row['topic']
        text = row['response_text']
        
        if chunk_size is None:
            rows.append({
                'chunk_id': doc_id,
                'doc_id': doc_id,
                'topic': topic,
                'text': text
            })
        else:
            chunks = chunk_text(text, chunk_size=chunk_size, overlap=overlap)
            for i, chunk in enumerate(chunks):
                rows.append({
                    'chunk_id': f"{doc_id}_chunk_{i}",
                    'doc_id': doc_id,
                    'topic': topic,
                    'text': chunk
                })
    
    return pd.DataFrame(rows)


def get_metrics(
    retrieved_docs: List[str],
    relevant_docs: List[str],
    metrics: List[str] = ['precision', 'recall', 'mrr', 'map']
) -> pd.DataFrame:

    relevant_set = set(relevant_docs)
    total_relevant = len(relevant_set)
    
    hits = sum(1 for doc in retrieved_docs if doc in relevant_set)
    
    result = {}
    
    if 'precision' in metrics:
        result['precision'] = hits / len(retrieved_docs) if len(retrieved_docs) > 0 else np.nan
    
    if 'recall' in metrics:
        result['recall'] = hits / total_relevant if total_relevant > 0 else np.nan
    
    if 'mrr' in metrics:
        first_relevant_rank = None
        for idx, doc_id in enumerate(retrieved_docs, start=1):
            if doc_id in relevant_docs:
                first_relevant_rank = idx
                break
        result['mrr'] = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank
    
    if 'map' in metrics:
        precisions = []
        hits_so_far = 0
        for i, doc in enumerate(retrieved_docs, 1):
            if doc in relevant_set:
                hits_so_far += 1
                precisions.append(hits_so_far / i)
        result['map'] = sum(precisions) / total_relevant if precisions and total_relevant > 0 else 0.0
    
    return pd.DataFrame([result])

In [4]:
class EmbeddingBackend:
    def __init__(self, model_name: str, device: str = "cpu", normalize: bool = True):
        self.model = SentenceTransformer(model_name, device=device)
        self.model_name = model_name
        self.normalize = normalize
    
    def encode(self, texts: List[str]) -> np.ndarray:
        vectors = self.model.encode(
            texts,
            batch_size=16,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=self.normalize
        )
        return vectors.astype("float32")

def build_embedding_backend(
    model_name: str = "paraphrase-multilingual-MiniLM-L12-v2",
    device: str = "cpu",
    normalize: bool = True
) -> EmbeddingBackend:
    try:
        backend = EmbeddingBackend(model_name=model_name, device=device, normalize=normalize)
        print(f"Модель: {model_name}, нормировка={normalize}")
        return backend
    except Exception as e:
        print(f"Ошибка загрузки {model_name}: {e}")
        raise

In [5]:
class VectorSearchIndex:
    def __init__(self, dim: int, similarity: str = "cosine"):

        self.dim = dim
        self.similarity = similarity
        self._faiss_index = None
        
        if similarity == "cosine":
            self._faiss_index = faiss.IndexFlatIP(dim)
        elif similarity == "euclidean":
            self._faiss_index = faiss.IndexFlatL2(dim)
        elif similarity == "dot":
            self._faiss_index = faiss.IndexFlatIP(dim)
    
    def add(self, vectors: np.ndarray) -> None:
        vectors = vectors.astype("float32")
        
        if self.similarity == "cosine":
            faiss.normalize_L2(vectors)
        
        self._faiss_index.add(vectors)
    
    def search(self, query_vectors: np.ndarray, top_k: int = 5) -> Tuple[np.ndarray, np.ndarray]:
        query_vectors = query_vectors.astype("float32")
        
        if self.similarity == "cosine":
            faiss.normalize_L2(query_vectors)
        
        scores, indices = self._faiss_index.search(query_vectors, top_k)
        
        if self.similarity == "euclidean":
            scores = -scores
        
        return scores, indices
    
def sanity_check_index(
    index: VectorSearchIndex,
    vectors: np.ndarray
):
    if index.similarity == "cosine":
        norms = np.linalg.norm(vectors, axis=1)
        print(f"Нормы документов: min={norms.min():.4f}, max={norms.max():.4f}")
        assert np.allclose(norms, 1.0, atol=1e-5), "Векторы не нормированы для cosine"

    print(f"{index.similarity}: векторы готовы для меры сходства, FAISS отработает корректно")

In [6]:
def evaluate_retrieval_bundle_pipeline(
    df_docs: pd.DataFrame,
    df_queries: pd.DataFrame,
    model_name: str,
    similarity: str,
    chunk_size: 40,
    overlap: int = 20,
    k: int = 5,
    device: str = "cpu"
) -> pd.DataFrame:

    print(f"Связка: модель={model_name.split('/')[-1]}, мера={similarity}, чанки={chunk_size if chunk_size else 'нет'}")

    
    df_chunks = buid_chunks(df_docs, chunk_size=chunk_size, overlap=overlap)
    print(f"Документов/чанков: {len(df_chunks)}")

    need_normalize = (similarity == "cosine")
    backend = build_embedding_backend(model_name, device=device, normalize=need_normalize)

    chunk_texts = df_chunks['text'].tolist()
    chunk_embeddings = backend.encode(chunk_texts)
    
    dim = chunk_embeddings.shape[1]
    index = VectorSearchIndex(dim, similarity=similarity)
    index.add(chunk_embeddings)
    
    sanity_check_index(index, chunk_embeddings)
    
    results = []
    
    for _, row in df_queries.iterrows():
        query = row['query_text']
        relevant_docs = row['relevant_docs'].split('|')
        
        query_vec = backend.encode([query])
        
        scores, indices = index.search(query_vec, top_k=k)
        
        predicted_chunk_ids = [df_chunks.iloc[idx]['chunk_id'] for idx in indices[0]]
        predicted_doc_ids = [cid.split('_chunk')[0] for cid in predicted_chunk_ids]
        
        metrics_df = get_metrics(
            retrieved_docs=predicted_doc_ids,
            relevant_docs=relevant_docs,
            metrics=['precision', 'recall', 'mrr', 'map']
        )
        
        results.append({
            'query_id': row['q_id'],
            'query': query,
            'relevant_docs': '|'.join(relevant_docs),
            'predicted_docs': '|'.join(predicted_doc_ids),
            'scores': '|'.join([f"{s:.4f}" for s in scores[0]]),
            'precision': metrics_df.iloc[0]['precision'],
            'recall': metrics_df.iloc[0]['recall'],
            'mrr': metrics_df.iloc[0]['mrr'],
            'map': metrics_df.iloc[0]['map']
        })
    
    return pd.DataFrame(results)



# Мера сходства

In [7]:
import logging
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)
df_docs = pd.DataFrame(pd.read_csv('../data/documents.csv'))
df_queries = pd.DataFrame(pd.read_csv('../data/queries.csv'))
doc_text_map = dict(zip(df_docs['doc_id'], df_docs['response_text']))

user2_configs = [
    {'name': 'USER2_cosine', 'similarity': 'cosine', 'chunk_size': None},
    {'name': 'USER2_dot', 'similarity': 'dot', 'chunk_size': None},
    {'name': 'USER2_euclidean', 'similarity': 'euclidean', 'chunk_size': None},
]

user2_results = []
best_mrr = -1
best_config = None
best_df = None

for cfg in user2_configs:
    display(Markdown(f"## {cfg['name']} | мера={cfg['similarity']}"))
        
    df_res = evaluate_retrieval_bundle_pipeline(
        df_docs=df_docs,
        df_queries=df_queries,
        model_name='deepvk/USER2-small',
        similarity=cfg['similarity'],
        chunk_size=cfg['chunk_size'],
        k=5,
        device='cpu'
    )
    
    df_res['first_predicted_id'] = df_res['predicted_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_predicted'] = df_res['first_predicted_id'].map(doc_text_map).fillna('')
    df_res['first_relevant_id'] = df_res['relevant_docs'].apply(lambda x: x.split('|')[0] if pd.notna(x) else '')
    df_res['first_hit'] = df_res['first_predicted_id'] == df_res['first_relevant_id']

    display(Markdown(f"### Результаты для {cfg['name']}"))
    display_cols = ['query', 'relevant_docs', 'predicted_docs', 'scores', 'first_predicted', 'first_hit', 'mrr', 'precision', 'recall', 'map']
    
    display(Markdown(f"### Топ 3 лучших результатов (по MRR)"))
    top3 = df_res.nlargest(3, 'mrr')
    display(top3[display_cols])

    display(Markdown(f"### Топ 3 худших результатов (по MRR)"))

    worst3 = df_res.nsmallest(3, 'mrr')
    display(worst3[display_cols])

    avg_precision = df_res['precision'].mean()
    avg_recall = df_res['recall'].mean()
    avg_mrr = df_res['mrr'].mean()
    avg_map = df_res['map'].mean()
    
    user2_results.append({
        'config': cfg['name'],
        'similarity': cfg['similarity'],
        'chunk_size': cfg['chunk_size'] if cfg['chunk_size'] else 'full',
        'precision': avg_precision,
        'recall': avg_recall,
        'mrr': avg_mrr,
        'map': avg_map,
    })

    

## USER2_cosine | мера=cosine

Связка: модель=USER2-small, мера=cosine, чанки=нет
Документов/чанков: 40


Loading weights: 100%|██████████| 74/74 [00:00<00:00, 10570.39it/s]


Модель: deepvk/USER2-small, нормировка=True
Нормы документов: min=1.0000, max=1.0000
cosine: векторы готовы для меры сходства, FAISS отработает корректно


### Результаты для USER2_cosine

...

### Результаты для USER2_euclidean

### Топ 3 лучших результатов (по MRR)

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map
3,Привет! забыл пароль что делать подскажите,doc_1|doc_3|doc_6,doc_1|doc_6|doc_7|doc_2|doc_8,-3.5946|-3.6464|-3.8718|-3.9731|-4.5222,Здравствуйте! Благодарим за обращение в нашу с...,True,1.0,0.4,0.666667,0.666667
8,Через сколько минут приходит письмо для сброса...,doc_1|doc_2|doc_7,doc_1|doc_2|doc_5|doc_3|doc_8,-4.5933|-4.8772|-5.2653|-5.2665|-5.3697,Здравствуйте! Благодарим за обращение в нашу с...,True,1.0,0.4,0.666667,0.666667
9,немогу зайти потму что пишет ошибка доступа,doc_1|doc_4|doc_5,doc_1|doc_4|doc_2|doc_36|doc_6,-5.8020|-6.2699|-6.3889|-6.4392|-6.5659,Здравствуйте! Благодарим за обращение в нашу с...,True,1.0,0.4,0.666667,0.666667


### Топ 3 худших результатов (по MRR)

,query,relevant_docs,predicted_docs,scores,first_predicted,first_hit,mrr,precision,recall,map
13,аплатил дважды верните деньги,doc_12|doc_13|doc_16,doc_1|doc_7|doc_6|doc_38|doc_14,-5.6081|-5.7520|-6.0210|-6.1127|-6.1326,Здравствуйте! Благодарим за обращение в нашу с...,False,0.0,0.0,0.0,0.0
16,скидку студенческую дайте яж студент,doc_10|doc_14|doc_16,doc_31|doc_25|doc_27|doc_18|doc_6,-5.9599|-6.1973|-6.6810|-6.8164|-7.0493,Приветствуем! Спасибо за обращение. Если курат...,False,0.0,0.0,0.0,0.0
23,сертифкат нескачивается с телефона помогите пж,doc_17|doc_20|doc_24,doc_25|doc_1|doc_7|doc_6|doc_8,-5.1234|-5.2271|-5.3614|-5.3942|-5.7042,Здравствуйте! Спасибо за обращение. Загрузка д...,False,0.0,0.0,0.0,0.0


In [8]:
summary_measure_df = pd.DataFrame(user2_results)

summary_cols = ['config', 'similarity', 'chunk_size', 'precision', 'recall', 'mrr', 'map']
summary_measure_df = summary_measure_df[summary_cols]

summary_measure_df = summary_measure_df.sort_values('mrr', ascending=False)

for col in ['precision', 'recall', 'mrr', 'map']:
    summary_measure_df[col] = summary_measure_df[col].round(4)

display(summary_measure_df)

best_row = summary_measure_df.iloc[0]

display(Markdown(f"### Лучшая конфигурация: {best_row['config']}"))
print(f"   MRR: {best_row['mrr']:.4f}")
print(f"   Precision: {best_row['precision']:.4f}")
print(f"   Recall: {best_row['recall']:.4f}")
print(f"   MAP: {best_row['map']:.4f}")

,config,similarity,chunk_size,precision,recall,mrr,map
0,USER2_cosine,cosine,full,0.3200,0.5217,0.7037,0.3936
2,USER2_euclidean,euclidean,full,0.2880,0.4700,0.6230,0.3461
1,USER2_dot,dot,full,0.2147,0.3500,0.4906,0.2474


### Лучшая конфигурация: USER2_cosine

   MRR: 0.7037
   Precision: 0.3200
   Recall: 0.5217
   MAP: 0.3936


In [9]:
summary_measure_df.to_csv(
    "../artifacts/user2_summary_measure_df.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Файл сохранён: ./artifacts/user2_summary_measure_df.csv")

Файл сохранён: ./artifacts/user2_summary_measure_df.csv
